# **Talented Lynxes: NYC Neighborhood Risk Recommendation System**

# Project Goal

Vibe Check helps NYC apartment hunters evaluate neighborhoods based on the urban issues they personally want to avoid.

Instead of only considering rent, commute, or amenities, our system asks:

**Given the urban problems a renter wants to avoid, which NYC areas show the strongest or weakest 311 complaint patterns for those concerns?**

The system combines:

- NYC 311 complaint data
- semantic matching with embeddings
- from-scratch clustering
- neighborhood ranking
- Streamlit visualization

## 1. System Overview

Our system follows a modular pipeline:

1. **Data preprocessing**  
   Clean NYC 311 records and keep the fields needed for recommendation.

2. **Semantic matching**  
   Match user concerns, such as "pests" or "loud noise", to official 311 complaint categories.

3. **Spatial clustering**  
   Cluster complaint locations using our own K-Means implementation.

4. **Ranking**  
   Rank clusters by how strongly they match the user's selected concerns.

5. **Visualization**  
   Display lower-concern areas and hotspots in the Streamlit app.

## 2. Data Source

We use the NYC 311 Service Requests dataset.

The full dataset covers:

**April 27, 2025 to April 27, 2026**

Important columns used in our pipeline:

- Created Date
- Problem
- Problem Detail
- Incident Zip
- Borough
- Latitude
- Longitude
- recency_weight

The dataset is too large to store fully in GitHub, so the repository includes a smaller sample file for testing.

In [ ]:
import pandas as pd

# Example structure based on our real NYC 311 sample data
sample_data = pd.DataFrame({
    "Created Date": ["3/13/2026 10:16", "3/20/2026 2:25", "3/20/2026 2:19"],
    "Agency": ["TLC", "DOT", "DOT"],
    "Problem": [
        "For Hire Vehicle Complaint",
        "Street Condition",
        "Street Condition"
    ],
    "Problem Detail": [
        "Driver Complaint - Non Passenger",
        "Pothole",
        "Pothole"
    ],
    "Incident Zip": ["10002", "11420", "11004"],
    "Borough": ["MANHATTAN", "QUEENS", "QUEENS"],
    "Latitude": [40.7209, 40.6722, 40.7417],
    "Longitude": [-73.9910, -73.8044, -73.7070]
})

sample_data

## 3. Preprocessing Logic


The preprocessing step cleans the raw 311 dataset and prepares it for matching and clustering.

Main steps:

1. Load raw 311 complaint data
2. Keep only the required columns
3. Remove records without valid latitude or longitude
4. Filter records to the selected one-year time window
5. Add a recency weight so newer complaints matter more

The recency weight is calculated as:

```text
recency_weight = exp(-lambda * delta_days)
lambda = 0.01


```python
# Simplified preprocessing pseudocode
# This is not meant to run the full preprocessing pipeline inside the notebook.

# df = pd.read_csv("data/raw/311_example_2.csv")

# Keep useful columns
# df = df[[
#     "Created Date",
#     "Problem",
#     "Problem Detail",
#     "Incident Zip",
#     "Borough",
#     "Latitude",
#     "Longitude"
# ]]

# Remove rows without valid location
# df = df.dropna(subset=["Latitude", "Longitude"])

# Convert Created Date to datetime
# df["Created Date"] = pd.to_datetime(df["Created Date"])

# Calculate recency weight
# df["recency_weight"] = np.exp(-0.01 * delta_days)

# Save processed data
# df.to_csv("data/processed/cleaned_raw_311.csv", index=False)

## 4. Semantic Matching

Users do not need to know the exact official 311 complaint category.

They can type natural language concerns, such as:

- "pests"
- "loud music at night"
- "broken heating"
- "potholes"
- "dirty streets"

The system maps these descriptions to structured 311 complaint categories.

### How it works

1. We combine `Problem` and `Problem Detail` into category labels.

   Example:

   ```text
   Street Condition - Pothole
   UNSANITARY CONDITION - PESTS
   Mobile Food Vendor - Insects / Pests

2. Each category label is converted into an embedding vector.

3. The user's query is also converted into an embedding vector.

4. The system calculates cosine similarity between the query vector and category vectors.

5. The top matching categories are returned.


```python
# Simplified semantic matching pseudocode

# category_labels = [
#     "Street Condition - Pothole",
#     "UNSANITARY CONDITION - PESTS",
#     "Mobile Food Vendor - Insects / Pests"
# ]

# category_embeddings = model.encode(category_labels)

# user_query = "pests"
# query_embedding = model.encode(user_query)

# similarities = cosine_similarity(query_embedding, category_embeddings)

# top_matches = select_top_k(category_labels, similarities, k=5)

## 5. Clustering

After matching complaint categories, we cluster the geographic locations of those complaints.

This project uses a from-scratch k-means implementation instead of calling `sklearn.cluster.KMeans`.

Main k-means steps:

1. Select initial centroids
2. Assign each complaint point to the nearest centroid
3. Recompute centroids based on assigned points
4. Repeat until the clusters stabilize
5. Use the final clusters to compare matched complaints against all local complaints

This allows us to identify both:

- areas where the user's concerns are relatively low
- hotspots where the user's concerns are relatively high

In [ ]:
# Simplified k-means pseudocode

# initialize centroids
centroids = choose_random_points(points, k)

for iteration in range(max_iter):
    assignments = assign_points_to_nearest_centroid(points, centroids)
    new_centroids = recompute_centroids(points, assignments)
    if centroids_stop_changing:
        break
    centroids = new_centroids

## 6. Ranking Formula

After clustering, each cluster receives a severity score.

The ranking uses both:

- how strongly complaints match the user's concern
- how recent those complaints are
- how many matched complaints appear in the cluster
- how large the local complaint baseline is

The main ranking formula is:

```text
severity_score = sum(recency_weight_i * similarity_i)

concern_share = severity_score / baseline_score

reliability_factor = log(1 + matched_complaint_count)

normalized_severity = concern_share * reliability_factor


```python
# Example ranking output

ranking_example = pd.DataFrame({
    "Rank": [1, 2, 3],
    "Borough": ["QUEENS", "BROOKLYN", "MANHATTAN"],
    "Zip Code": ["11375", "11215", "10003"],
    "Concern Share": [0.03, 0.05, 0.12],
    "Reliability Factor": [2.7, 3.1, 4.0],
    "Normalized Severity": [0.081, 0.155, 0.480],
    "Result Type": ["Lower-Concern", "Lower-Concern", "Hotspot"]
})

ranking_example

## 7. Streamlit App Interface

The Streamlit app provides the interactive user experience.

Main user flow:

1. User enters a concern
2. App returns matching 311 complaint categories
3. User selects concerns to include
4. User chooses clustering method
5. App generates:
   - ranked lower-concern areas
   - ranked hotspots
   - interactive map
   - Zillow links for lower-concern areas


```python
# To run the app locally:
# streamlit run app.py

## 8. Example User Scenario

Suppose a renter wants to avoid pest-related problems.

They type:

```text
pests

The system maps this input to official 311 categories such as:

- UNSANITARY CONDITION - PESTS
- Mobile Food Vendor - Insects / Pests
- Food Establishment - Rodents/Insects/Garbage

Then the system clusters related complaints geographically and ranks NYC areas.

The final result helps the renter identify:

- areas with fewer pest-related complaints
- hotspots with stronger pest-related complaint patterns
- neighborhoods that may require more caution before signing a lease


```markdown
## 9. Team Contributions

- **Yujia Guo**  
  Data collection and preprocessing of NYC 311 records.

- **Isabella Liu**  
  Repository organization, README/design document, requirements/setup, and checkpoint submission polishing.

- **Zeyue Xu**  
  Clustering, neighborhood ranking, and evaluation.

- **Joseph Zhang**  
  Frontend/backend integration and Streamlit app structure.

- **Jinyu Zheng**  
  Embedding-based complaint matching and similarity pipeline.

## 10. Limitations

Although the system is useful for neighborhood-level exploration, it has some limitations:

1. **311 data is complaint-based**  
   It reflects reported issues, not all real issues.

2. **Complaint volume may be biased**  
   Some neighborhoods may report problems more frequently than others.

3. **Sample data is smaller**  
   The sample dataset is good for testing, but the full dataset gives more reliable rankings.

4. **Neighborhood quality is complex**  
   The app focuses on selected urban stressors, not every factor that matters in apartment hunting.

These limitations are important because the system should support decision-making, not replace personal judgment or further research.

## 11. Main Takeaways

Vibe Check turns subjective renter preferences into measurable neighborhood signals.

The project demonstrates:

- practical use of public urban data
- semantic matching with embeddings
- from-scratch implementation of clustering
- personalized neighborhood ranking
- an interactive web app for real users

The final result is not just a map of complaints, but a personalized neighborhood recommendation system based on what the user wants to avoid.